# Test Queries 

In [ ]:
import pandas as pd
import json
import os
import psycopg2
from psycopg2.extras import execute_values
import requests

from pathlib import Path

In [ ]:

DB_HOST=os.environ.get('DB_HOST')
DB_NAME=os.environ.get('DB_NAME')
DB_USER=os.environ.get('DB_USER')
DB_PASSW=os.environ.get('DB_PASSWORD')
DB_PORT=os.environ.get('DB_PORT')


In [ ]:
#Connect to database citikike
try:
    conn = psycopg2.connect(
            host = DB_HOST,
            database = DB_NAME,
            user = DB_USER,
            password = DB_PASSW, 
            port = DB_PORT,
            keepalives=1,
            keepalives_idle=30,
            keepalives_interval=10,
            keepalives_count=5
    )
    print('Connection Successful')

    cursor = conn.cursor()
except Exception as e:
    print(f'Could not establish connection:{e}')

### Query 1: Stations basic lookup with flags and status and recent trip activities in the last 7 days 

In [ ]:
cursor.execute('''select count(*) from station_status_changes ''')
cursor.fetchone()

In [ ]:
cursor.execute('''
SELECT s.station_id,
    s.name,
	s.short_name,
    s.latitude,
  	s.longitude,
  	s.capacity,
 	s.is_active,
COALESCE(rf.flag_type, 'none') AS flag_status,
COALESCE(rf.severity,'none') AS flag_severity,
COALESCE(rf.detected_at) AS detected_at,
(SELECT COUNT(*) FROM trips t
    WHERE t.start_station_id = s.short_name AND
    t.started_at >= (SELECT MAX(started_at) FROM trips) - INTERVAL '7days') AS trips_last_7_days
FROM stations s
LEFT JOIN rebalancing_flags rf ON rf.station_id = s.station_id
WHERE s.station_id =  '66db41bf-0aca-11e7-82f6-3863bb44ef7c' --:station;
''')
cursor.fetchall()

In [ ]:
#find stations by name:
SELECT station_id, short_name, station_name, latitude, longitude
FROM stations
WHERE station_name ILIKE :search_term
ORDER BY station_name
LIMIT 20

### Query 2 Availability by station showing current status 


In [ ]:
cursor.execute('''
SELECT 
    a.station_id,
    s.name,
    a.num_bikes_available,
    a.num_ebikes_available,
    a.num_docks_available, 
    a.captured_at,
    s.capacity,
    CAST(100.0 * num_bikes_available /  NULLIF(s.capacity,0) AS DECIMAL(5,1)) AS pct_full,
    CASE
        WHEN a.num_bikes_available = 0 THEN 'empty'
        WHEN a.num_docks_available = 0 THEN 'full'
        WHEN a.num_bikes_available < 4 THEN 'low'
        ELSE 'normal'
    END AS availability_status
FROM availability_snapshots a
JOIN stations s ON a.station_id = s.station_id
WHERE s.station_id = '66db41bf-0aca-11e7-82f6-3863bb44ef7c' --:station_id
ORDER BY a.captured_at DESC 
LIMIT 1;
  
  
''')
cursor.fetchone()

### Query 3: Rolling 7-day metrics: daily_station_metrics with window function for rolling average over 7 days



In [ ]:
cursor.execute('''
SELECT
    station_id, 
    summary_date,
    trips_started,
    trips_ended, 
    net_flow,
    pct_time_empty, 
    pct_time_full, 
    ROUND(AVG(trips_started) OVER (ORDER BY summary_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW),2) AS trips_started_rolling_avg,
    LAG(trips_started,1) OVER (ORDER BY summary_date) AS pervious_day_starts,
    CASE
        WHEN trips_started > LAG(trips_started,1) OVER (ORDER BY summary_date) THEN 'up'
        WHEN trips_started < LAG(trips_started,1) OVER (ORDER BY summary_date) THEN 'down'
        ELSE 'same as day before'
        END AS trend

FROM daily_station_metrics
WHERE station_id = '22ab59a8-0af4-4101-9350-a7af88e37c2e'
    AND summary_date >= (SELECT MAX(summary_date) FROM daily_station_metrics WHERE station_id = '22ab59a8-0af4-4101-9350-a7af88e37c2e') - INTERVAL '15 days'
ORDER BY summary_date DESC;
''')
cursor.fetchall()


In [ ]:
cursor.execute('''SELECT MAX(summary_date) FROM daily_station_metrics WHERE station_id = '22ab59a8-0af4-4101-9350-a7af88e37c2e'
   ;''')
cursor.fetchall()

### Query 4 Hourly demand patterns

In [ ]:
cursor.execute('''
SELECT 
    hour_of_day,
    CASE 
        WHEN day_of_week BETWEEN 1 AND 5 THEN 'weekday'
        ELSE 'weekend' END AS weekday_weekend,
    ROUND(AVG(avg_trips_started),2),
    ROUND(AVG(avg_trips_ended),2),
    ROUND(AVG(avg_net_flow),2)
FROM hourly_patterns
WHERE station_id = '22ab59a8-0af4-4101-9350-a7af88e37c2e'
GROUP BY hour_of_day,    
    CASE 
        WHEN day_of_week BETWEEN 1 AND 5 THEN 'weekday'
        ELSE 'weekend'
    END 
ORDER BY hour_of_day, weekday_weekend;
''')
cursor.fetchall()

### Query 5: Top outbound destinations:station_pairs table, where do riders go from this station



In [ ]:
cursor.execute('''
SELECT sp.end_station_id,
    s.name AS destination_name,
    sp.trip_count,
    sp.avg_duration_seconds,
    ROUND(100 * sp.member_trips / NULLIF(sp.trip_count,0),2) AS pct_member_trips
FROM station_pairs sp
JOIN stations s ON s.short_name = sp.end_station_id
WHERE sp.start_station_id = '7925.04'
ORDER BY sp.trip_count DESC
LIMIT 10;
                    ''')
cursor.fetchall()

### Querty 6: Top inbound origins:station_pairs + stations: where do riders come from to this station

In [ ]:
cursor.execute('''
SELECT sp.start_station_id,
    s.name AS origin_name,
    sp.trip_count,
    sp.avg_duration_seconds,
    ROUND(100 * sp.member_trips / NULLIF(sp.trip_count,0),2) AS pct_member_trips
FROM station_pairs sp
JOIN stations s ON s.short_name = sp.start_station_id
WHERE sp.end_station_id = '7925.04'
ORDER BY sp.trip_count DESC
LIMIT 10;
                    ''')
cursor.fetchall()

### Query 7: Recent status changes

In [ ]:
#print columns of station_status_changes table
cursor.execute('''SELECT COLUMN_NAME
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_NAME = 'station_status_changes';
                    ''')
cursor.fetchall()

In [ ]:
#see count and types of change_type
cursor.execute('''
select change_type, count(*)
from station_status_changes
group by change_type;
''')
cursor.fetchall()

In [ ]:
#Query 7
cursor.execute('''
SELECT 
    change_type,
    previous_value,
    detected_at,
    detected_at - LAG(detected_at) OVER (ORDER BY detected_at) AS time_since_last_change,
    CASE 
        WHEN change_type IN ('became_empty', 'became_full', 'went offline') THEN 'problem'
        WHEN change_type IN ('recovered_from_full','recovered from empty','came online') THEN 'recovered'
    END AS event_category
FROM station_status_changes
WHERE station_id = '22ab59a8-0af4-4101-9350-a7af88e37c2e' --:station_id
ORDER BY detected_at DESC LIMIT 5;
''')
cursor.fetchall()

In [ ]:
#check count of detected_at dates for this station
cursor.execute('''
 SELECT detected_at, count(*)
 from station_status_changes
 where station_id = '22ab59a8-0af4-4101-9350-a7af88e37c2e'
 group by detected_at;
''')
cursor.fetchall()

### Query 8: Flagged stations list

In [ ]:
#print columns of rebalancing_flags table
cursor.execute('''
SELECT COLUMN_NAME
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_NAME = 'rebalancing_flags'
''')
cursor.fetchall()

In [ ]:
#count flag_type
cursor.execute('''
SELECT flag_type, count(*)
FROM rebalancing_flags
GROUP BY flag_type
''')
cursor.fetchall()

In [ ]:
#count severity
cursor.execute('''
SELECT severity, count(*)
FROM rebalancing_flags
GROUP BY severity
''')
cursor.fetchall()

In [ ]:
#Query 8
cursor.execute('''
SELECT s.station_id,
    s.name,
    s.short_name,
    rf.flag_type,
    rf.severity,
    s.latitude,
    s.longitude,
    CASE 
        WHEN resolved_at IS NULL THEN 'active'
        ELSE 'resolved'
    END AS resolution_status,
    s.capacity
 
FROM rebalancing_flags rf
LEFT JOIN stations s ON rf.station_id = s.station_id
WHERE (rf.flag_type = :flag_type OR :flag_type IS NULL)
    AND (rf.severity= :severity OR :severity IS NULL)
    AND resolved_at IS NULL --active
ORDER BY 
   CASE 
        WHEN rf.severity = 'high' THEN 1
        WHEN rf.severity = 'medium' THEN 2
        ELSE 3
    END,
    detected_at DESC
    LIMIT :limit;

''')
cursor.fetchall()

### Query 9: Flag summary counts

In [ ]:
cursor.execute('''
SELECT 
    flag_type,
    severity,
    COUNT(*) AS active_count,
    MIN(detected_at) AS oldest_active,
    MAX(detected_at) AS newest_active
FROM rebalancing_flags
WHERE resolved_at IS NULL
GROUP BY flag_type, severity
ORDER BY flag_type,
    CASE 
        WHEN severity = 'high' THEN 1
        WHEN severity = 'medium' THEN 2
        ELSE 3
    END;
''')
cursor.fetchall()

### Query 10 system stats

In [ ]:
cursor.execute('''
SELECT
    (SELECT COUNT(*) FROM stations) AS total_stations,
    (SELECT COUNT(*) FROM stations WHERE is_active = 'true') AS active_stations,
    (SELECT COUNT(*) FROM trips) AS total_trips,
    (SELECT COUNT(*) FROM availability_snapshots) AS total_snapshots,
    (SELECT COUNT(*) FROM rebalancing_flags WHERE resolved_at IS NULL) AS active_flags,
    (SELECT COUNT(*) FROM rebalancing_flags WHERE flag_type = 'chronic_empty' AND resolved_at IS NULL) AS active_chronic_empty,
    (SELECT COUNT(*) FROM rebalancing_flags WHERE flag_type = 'chronic_full' AND resolved_at IS NULL) AS active_chronic_full;
''')
cursor.fetchone()

### Query 11  Data date range: MIN/MAX from trips, availability_snapshots 

In [ ]:
cursor.execute('''select * from rebalancing_flags limit 3''')
cursor.fetchall()

In [ ]:
cursor.execute('''
SELECT 
    (SELECT MIN(captured_at) FROM availability_snapshots) AS snapshot_starts,
        (SELECT MAX(captured_at) FROM availability_snapshots) AS snapshot_ends,
        (SELECT MIN(started_at) FROM trips) AS trips_start,
        (SELECT MAX(started_at) FROM trips) AS trips_end
''')
cursor.fetchone()

### Query 12  Busiest corridors of station_pair

In [ ]:
cursor.execute('''
SELECT 
    sp.start_station_id,
    s1.name AS start_station_name,
    sp.end_station_id,
    s2.name AS end_station_name,
    sp.trip_count,
    sp.avg_duration_seconds,
    ROUND(sp.avg_duration_seconds / 60.0,2) AS trip_duratin_min,
    ROUND(100.0 * sp.member_trips / NULLIF(sp.trip_count,0),2) AS pct_member
FROM station_pairs sp
JOIN stations s1 ON s1.short_name = sp.start_station_id
JOIN stations s2 ON s2.short_name = sp.end_station_id
WHERE trip_count >= 100
ORDER BY sp.trip_count DESC
LIMIT 2
''')
cursor.fetchall()

### Query 13  Bidirectional trip imbalance 

In [ ]:
cursor.execute('''
SELECT 
    LEAST(sp1.start_station_id, sp1.end_station_id) AS station_a,
    GREATEST(sp1.start_station_id, sp1.end_station_id) AS station_b,
    s1.name AS station_a_name,
    s2.name AS station_b_name,
    COALESCE(sp1.trip_count,0) AS trips_a_to_b,
    COALESCE(sp2.trip_count,0) AS trips_b_to_a,
    COALESCE(sp1.trip_count,0)+ COALESCE(sp2.trip_count,0) AS total_trips,
    ABS(COALESCE(sp1.trip_count,0)- COALESCE(sp2.trip_count,0)) AS imbalance,
    ROUND(100.0 * ABS(COALESCE(sp1.trip_count,0)- COALESCE(sp2.trip_count,0)) /
     NULLIF(COALESCE(sp1.trip_count,0)+ COALESCE(sp2.trip_count,0),0), 2) AS pct_imbalance,
     CASE 
         WHEN COALESCE(sp1.trip_count,0) > COALESCE(sp2.trip_count,0) THEN 'toward ' || s2.name
         WHEN COALESCE(sp1.trip_count,0) < COALESCE(sp2.trip_count,0) THEN 'toward ' || s1.name
         ELSE 'balanced'
    END AS imbalance_direction
FROM station_pairs sp1
LEFT JOIN station_pairs sp2 ON 
        sp1.start_station_id = sp2.end_station_id AND
        sp1.end_station_id = sp2.start_station_id
JOIN stations s1 ON LEAST(sp1.start_station_id,sp1.end_station_id) = s1.short_name  
JOIN stations s2 ON GREATEST(sp1.start_station_id,sp1.end_station_id) = s2.short_name 
WHERE sp1.start_station_id < sp1.end_station_id
ORDER BY ABS(COALESCE(sp1.trip_count,0)- COALESCE(sp2.trip_count,0)) DESC
limit 3;
''')
cursor.fetchall()

### Query 14 Problem stations ranked

In [ ]:
cursor.execute('''
SELECT 
	s.station_id,
    s.short_name,
	s.name,
    rf.flag_type,
    rf.severity,
    rf.detected_at,  
	RANK() OVER (PARTITION BY rf.flag_type ORDER BY 
                                    CASE
                                        WHEN severity = 'high' THEN 1
                                        WHEN severity = 'medium' THEN 2
                                        ELSE 3
                                    END
                                    , rf.detected_at) AS rank_within_flag,
    COUNT(*) OVER (PARTITION BY rf.flag_type) as total_in_flag_type

FROM rebalancing_flags rf
JOIN stations s ON s.station_id = rf.station_id
WHERE rf.resolved_at IS NULL
ORDER BY rf.flag_type, rank_within_flag limit 5;
''')
cursor.fetchall()

### Query 15 Stations with worst availability

In [ ]:
cursor.execute('''
SELECT s.station_id,
    s.short_name,
    s.name,
    s.capacity,
    ROUND(AVG(dsm.pct_time_empty),2) AS avg_empty,
    ROUND(AVG(dsm.pct_time_full),2) AS avg_full,
    ROUND(AVG(dsm.pct_time_empty) + AVG(dsm.pct_time_full),2) AS avg_pct_unavailable,
    COUNT(*) AS days_sampled
FROM daily_station_metrics dsm
JOIN stations s ON s.station_id = dsm.station_id
WHERE summary_date >= (SELECT MAX(summary_date) FROM daily_station_metrics) - INTERVAL '30 days'
GROUP BY s.station_id, s.short_name, s.name, s.capacity
HAVING AVG(dsm.pct_time_empty) + AVG(dsm.pct_time_full) >
                        (SELECT AVG(pct_time_empty + pct_time_full) FROM daily_station_metrics)
ORDER BY  avg_pct_unavailable DESC
LIMIT 20;
''')
cursor.fetchall()

In [ ]:
conn.close()